# Topic 21: Transfer Learning: Adapting a Pretrained CNN to Your Own Images

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 21 lecture note](https://dongchul.kim/csci3320-fall2026/topic/23), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

**Use a GPU for this one**: Runtime > Change runtime type > T4 GPU (free). Way 2,
fine-tuning, takes roughly 5 to 10 minutes on a T4 and hours on a CPU. Numbers can differ a little from the note on a
different library version.

## Part 1: the data, and only 5,000 labeled images

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
train = torchvision.datasets.CIFAR10("data", train=True, download=True)
test = torchvision.datasets.CIFAR10("data", train=False, download=True)
X_train = torch.tensor(train.data).permute(0, 3, 1, 2).float() / 255   # (50000, 3, 32, 32)
X_test = torch.tensor(test.data).permute(0, 3, 1, 2).float() / 255     # (10000, 3, 32, 32)
y_train, y_test = torch.tensor(train.targets), torch.tensor(test.targets)

keep = torch.randperm(50000)[:5000]          # pretend we only have 5,000 labeled images
X_small, y_small = X_train[keep], y_train[keep]

print(train.data.shape, "->", X_train.shape)
print("we keep:", X_small.shape, "  test:", X_test.shape)
print(train.classes)
print("device:", device)

## Part 2: the baseline, Topic 20's CNN from scratch

In [ ]:
def train_and_test(model, X, y, epochs, lr, prepare=lambda x: x):
    model = model.to(device)
    loader = DataLoader(TensorDataset(X, y), batch_size=128, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = prepare(xb.to(device)), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        pred = torch.cat([model(prepare(X_test[i:i + 500].to(device))).argmax(1).cpu()
                          for i in range(0, len(X_test), 500)])
    return (pred == y_test).float().mean().item()

In [ ]:
torch.manual_seed(0)
scratch = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 3 channels in: color
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16 * 8 * 8, 10),                                      # 32 -> 16 -> 8
)
print("weights:", sum(p.numel() for p in scratch.parameters()))
acc = train_and_test(scratch, X_small, y_small, epochs=20, lr=0.001)
print(f"from scratch, 5,000 images: test accuracy {acc:.4f}")

## Part 3: a network someone else trained

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
net = resnet18(weights=weights)
print("weights:", sum(p.numel() for p in net.parameters()))
print("trained on ImageNet:", len(weights.meta["categories"]), "classes")
print("its ImageNet validation accuracy:", weights.meta["_metrics"]["ImageNet-1K"]["acc@1"], "%")
print("its last layer:", net.fc)

## Part 4: give it what it expects

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

def prepare(x):
    x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)   # 32 x 32 -> 224 x 224
    return (x - MEAN) / STD                                                # ImageNet's mean and std

x = X_test[:4].to(device)
print(x.shape, "->", prepare(x).shape)

### Ask it what it sees, before we change anything

In [ ]:
net = net.to(device).eval()
with torch.no_grad():
    probs = net(prepare(X_test[:4].to(device))).softmax(dim=1)
top = probs.topk(3, dim=1)
for i in range(4):
    guesses = ", ".join(f"{weights.meta['categories'][j]} {p:.2f}"
                        for p, j in zip(top.values[i].tolist(), top.indices[i].tolist()))
    print(f"{test.classes[y_test[i]]:6s} -> {guesses}")

## Way 1: a feature maker

### Step 1: turn every image into 512 numbers

In [ ]:
body = resnet18(weights=weights)
body.fc = nn.Identity()            # no last layer: the output is the 512 numbers before it
body = body.to(device).eval()

@torch.no_grad()
def features(X):
    return torch.cat([body(prepare(X[i:i + 250].to(device))).cpu() for i in range(0, len(X), 250)])

F_small = features(X_small)
F_test = features(X_test)
print(X_small.shape, "->", F_small.shape)
print(X_test.shape, "->", F_test.shape)

### Step 2: train only a new last layer

In [ ]:
torch.manual_seed(0)
head = nn.Sequential(nn.Linear(512, 10)).to(device)
loader = DataLoader(TensorDataset(F_small, y_small), batch_size=128, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(head.parameters(), lr=0.001)
for epoch in range(20):
    for fb, yb in loader:
        fb, yb = fb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(head(fb), yb)
        loss.backward()
        optimizer.step()
with torch.no_grad():
    pred = head(F_test.to(device)).argmax(1).cpu()
print("weights trained:", sum(p.numel() for p in head.parameters()))
print(f"frozen ResNet18 + new last layer, 5,000 images: test accuracy {(pred == y_test).float().mean():.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000)
clf.fit(F_small.numpy(), y_small.numpy())
print(f"LogisticRegression on the 512 features: test accuracy {clf.score(F_test.numpy(), y_test.numpy()):.4f}")

## Way 2: fine-tuning

In [ ]:
torch.manual_seed(0)
model = resnet18(weights=weights)
model.fc = nn.Linear(512, 10)          # 1000 ImageNet classes -> our 10
acc = train_and_test(model, X_small, y_small, epochs=20, lr=0.0001, prepare=prepare)
print(f"fine-tuned ResNet18, 5,000 images: test accuracy {acc:.4f}")

### Freezing inside one model

In [ ]:
model = resnet18(weights=weights)
for p in model.parameters():
    p.requires_grad = False            # freeze every weight it came with
model.fc = nn.Linear(512, 10)          # a new layer: its weights are trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} of {total:,}")

## The error you will see: a gray image has 1 channel

In [ ]:
gray = torch.rand(8, 1, 28, 28).to(device)          # 8 Fashion-MNIST-sized images
try:
    body(gray)
except RuntimeError as e:
    print("RuntimeError:", str(e))
print("fixed:", body(prepare(gray.repeat(1, 3, 1, 1))).shape)

Back to the [Topic 21 lecture note](https://dongchul.kim/csci3320-fall2026/topic/23).